In [1]:
from math import sqrt
import pandas as pd
import numpy as np

## Programación de los test de Hipótesis

In [2]:
import scipy

##Test de Welch

#Estadístico T
def T(X,Y):
    n = len(X)
    m = len(Y)

    numerador = X.mean() - Y.mean()

    denominador = sqrt((X.var(ddof = 1)/n)+(Y.var(ddof = 1)/m))

    return numerador/denominador


#Nu (grados de libertad de la distribución)

def nu(X,Y):
    n = len(X)
    m = len(Y)

    S2X_n = X.var(ddof = 1)/n
    S2Y_m = Y.var(ddof = 1)/m


    numerador = (S2X_n + S2Y_m)**2

    denominador = S2X_n**2 /(n-1) + S2Y_m**2 / (m-1)

    return numerador / denominador

#Test de Welch

def Welch(X,Y, alpha):
    est = T(X,Y)
    Nu = nu(X,Y)

    p = 2*(1- scipy.stats.t.cdf(abs(est), df=Nu))

    R = abs(est) >= scipy.stats.t.ppf(1-alpha/2, df=Nu)

    return (est,p,R)

In [3]:
##Test Wilcoxon

def Wilcoxon(X,Y, alpha):


    #Creo una tabla que diga su rango junto a grupo incial x o y:
    datos = np.concatenate([X, Y])
    grupos = ["x"] * len(X) + ["y"] * len(Y)

    df = pd.DataFrame({"Dato": datos, "Grupo": grupos})

    df = df.sort_values(by="Dato").reset_index(drop=True)

    df["rango"] = df.index + 1

    W = df[df["Grupo"] == "x"]["rango"].sum()

    return (W,"p-value","Bool") #NO SE CALCULAR LA REGION AYUDA

In [4]:
#Test Mann–Whitney }


def MannWhitney(X,Y, alpha):

    n = len(X)
    m = len(Y)
    N = n + m

    #Creo una tabla que diga su rango junto a grupo incial x o y:
    datos = np.concatenate([X, Y])
    grupos = ["x"] * n + ["y"] * m

    df = pd.DataFrame({"Dato": datos, "Grupo": grupos})

    df["rango"] = df["Dato"].rank()

    #Calculo de los estadísticos principales
    W_x = df[df["Grupo"] == "x"]["rango"].sum()
    W_y = df[df["Grupo"] == "y"]["rango"].sum()

    U_x = n*m + n*(n+1)/2 - W_x

    U_y = n*m + m*(m+1)/2 - W_y

    U = min(U_x,U_y)

    E = n*m/2

    #Agrupa los datos, anota la frecuencia y rescatamos el array de repeticiones
    t = df.groupby("Dato").size().to_numpy()

    suma = np.sum(t**3 -t)

    var_U = (N+1 - (suma/(N*(N-1))))*E/6

    epsilon = (U-E)/sqrt(var_U)

    Rechazo = abs(epsilon) >= scipy.stats.norm.ppf(1-alpha/2)

    p = 2*(1- scipy.stats.norm.cdf(abs(epsilon)))





    return (epsilon,p, Rechazo)

## Aplicación 
### Ajustamos el dataframe desplazando los puntajes para que empiecen en cero

In [5]:
df_raw = pd.read_csv("../data/raw/colegios_independencia_PAES2025.csv")

df_raw.columns

Index(['ID_aux', 'RBD', 'NOMBRE_COLEGIO', 'TIPO_ESTABLECIMIENTO', 'MODALIDAD',
       'PTJE_NEM', 'PTJE_RANKING', 'CLEC_REG_ACTUAL', 'MATE1_REG_ACTUAL',
       'MATE2_REG_ACTUAL', 'HCSOC_REG_ACTUAL', 'CIEN_REG_ACTUAL',
       'MODULO_REG_ACTUAL'],
      dtype='str')

In [6]:
df_raw.describe()

,RBD,PTJE_NEM,PTJE_RANKING,CLEC_REG_ACTUAL,MATE1_REG_ACTUAL,MATE2_REG_ACTUAL,HCSOC_REG_ACTUAL,CIEN_REG_ACTUAL
count,1114.000000,1072.000000,1072.000000,723.000000,922.000000,398.000000,479.000000,632.000000
mean,9670.301616,698.412313,711.545709,578.348548,584.057484,387.670854,469.392484,462.914557
std,2476.017181,145.365914,156.986210,104.721793,130.942080,78.405776,116.511786,96.295532
min,8497.000000,275.000000,275.000000,402.000000,331.000000,220.000000,206.000000,267.000000
25%,8589.000000,594.000000,594.000000,494.000000,496.000000,344.000000,385.000000,404.000000
50%,8616.000000,703.000000,715.000000,577.000000,557.000000,378.000000,444.000000,440.000000
75%,10223.000000,814.000000,837.000000,660.000000,648.000000,405.000000,534.500000,506.000000
max,24763.000000,1000.000000,1000.000000,954.000000,992.000000,780.000000,917.000000,828.000000


In [7]:
df = df_raw.copy()

Desplazar = ['PTJE_NEM', 'PTJE_RANKING', 'CLEC_REG_ACTUAL', 'MATE1_REG_ACTUAL',
       'MATE2_REG_ACTUAL', 'HCSOC_REG_ACTUAL', 'CIEN_REG_ACTUAL']

df[Desplazar] = df[Desplazar] - 100

In [8]:
df.to_csv("../data/precessed/colegios_independencia_PAES2025_desplazados.csv")

In [9]:
df.describe()

,RBD,PTJE_NEM,PTJE_RANKING,CLEC_REG_ACTUAL,MATE1_REG_ACTUAL,MATE2_REG_ACTUAL,HCSOC_REG_ACTUAL,CIEN_REG_ACTUAL
count,1114.000000,1072.000000,1072.000000,723.000000,922.000000,398.000000,479.000000,632.000000
mean,9670.301616,598.412313,611.545709,478.348548,484.057484,287.670854,369.392484,362.914557
std,2476.017181,145.365914,156.986210,104.721793,130.942080,78.405776,116.511786,96.295532
min,8497.000000,175.000000,175.000000,302.000000,231.000000,120.000000,106.000000,167.000000
25%,8589.000000,494.000000,494.000000,394.000000,396.000000,244.000000,285.000000,304.000000
50%,8616.000000,603.000000,615.000000,477.000000,457.000000,278.000000,344.000000,340.000000
75%,10223.000000,714.000000,737.000000,560.000000,548.000000,305.000000,434.500000,406.000000
max,24763.000000,900.000000,900.000000,854.000000,892.000000,680.000000,817.000000,728.000000


### Separamos tres pares de arrays para comparar

Compararemos los puntajes de la prueba competencia de matemáticas 1 regular para estudiantes con modalidad diurna,

Test 1 Welch: Compararemos la media entre los estudiantes en establecimientos municipales y los estudiantes en establecimientos particulares (subvencionado o pagado).

Test 2 Welch: Compararemos la media entre los estudiantes que rindieron M2 y los que no.

Test 3 Test Mann–Whitney: La probabilidad de que un estudiante que tuvo sobre la mediana en la prueba competencia lectora obtenga mejor puntaje que a uno bajo la mediana es igual a que obtenga peor puntaje (con desigualdad estricta).

In [10]:
#TEST 1

X1 = df[(df["TIPO_ESTABLECIMIENTO"] == "Municipal") & (df["MODALIDAD"] == "Diurna") & (df["MATE1_REG_ACTUAL"].notna())]["MATE1_REG_ACTUAL"].to_numpy()
Y1 = df[(df["TIPO_ESTABLECIMIENTO"] != "Municipal") & (df["MODALIDAD"] == "Diurna") & (df["MATE1_REG_ACTUAL"].notna())]["MATE1_REG_ACTUAL"].to_numpy()

#TEST 2

X2 = df[(df["MATE2_REG_ACTUAL"].isna()) & (df["MODALIDAD"] == "Diurna") & (df["MATE1_REG_ACTUAL"].notna())]["MATE1_REG_ACTUAL"].to_numpy()
Y2 = df[(df["MATE2_REG_ACTUAL"].notna()) & (df["MODALIDAD"] == "Diurna") & (df["MATE1_REG_ACTUAL"].notna())]["MATE1_REG_ACTUAL"].to_numpy()

#Test 3

medleng = df[df["MODALIDAD"]== "Diurna"]["CLEC_REG_ACTUAL"].median()

X3 = df[(df["CLEC_REG_ACTUAL"] > medleng) & (df["MODALIDAD"] == "Diurna") & (df["MATE1_REG_ACTUAL"].notna())]["MATE1_REG_ACTUAL"].to_numpy()
Y3 = df[(df["CLEC_REG_ACTUAL"] < medleng) & (df["MODALIDAD"] == "Diurna") & (df["MATE1_REG_ACTUAL"].notna())]["MATE1_REG_ACTUAL"].to_numpy()

In [11]:
#Test 1
alpha = 0.01
R1 = Welch(X1, Y1, alpha)

print(R1)

(np.float64(-4.293117835522048), np.float64(2.182712132436926e-05), np.True_)


In [12]:
#Test 2
alpha = 0.01
R2 = Welch(X2, Y2, alpha)

print(R2)

(np.float64(-6.628959523927556), np.float64(6.797140628123088e-11), np.True_)


In [13]:
#Test 3
alpha = 0.05
R3 = MannWhitney(X3,Y3, alpha)

print(R3)

(np.float64(-15.079297178218747), np.float64(0.0), np.True_)
